In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_REV = '28a4f10a0e9b9fc5708ce12d1f862dac5f1f0a04'
REPO_ROOT = Path('/kaggle/working/spider')
subprocess.run(['git', 'clone', 'https://github.com/yogesh-dhande/spider.git', str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_REV], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '300'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'


In [ ]:
%pip install -q --progress-bar off -r requirements/experiment2-kaggle.txt


In [ ]:
from spider.exp4_data import find_exp4_checkpoint, find_exp4_data
from spider.workflow import gpu_summary

prepared = find_exp4_data('/kaggle/input')
adapter = find_exp4_checkpoint('/kaggle/input', 1875)
os.environ['SPIDER_DATA_DIR'] = str(prepared)
print({'event': 'validation_inputs', 'prepared': str(prepared), 'adapter': str(adapter), **gpu_summary()}, flush=True)


In [ ]:
from spider.action_evaluate import evaluate_actions

action_predictions_path, action_metrics = evaluate_actions('configs/experiment4.yaml', 'action-development-step-1875', str(adapter), split='development')
print({'event': 'action_development_complete', 'metrics': action_metrics}, flush=True)


In [ ]:
import torch

torch.cuda.empty_cache()
from spider.probe import run_validation_probe

probe_path = run_validation_probe('configs/experiment4.yaml', 'perception-development-step-1875', str(adapter), step=1875, limit_per_task=128)
perception = json.loads(probe_path.read_text())
print({'event': 'perception_development_complete', 'metrics': perception['primary_metrics']}, flush=True)


In [ ]:
from spider.dashboard import (
    build_probe_dashboard,
    copy_action_dashboard_images, write_dashboard_json,
)

action_baseline_predictions = list(
    Path('/kaggle/input').rglob('action-exp002/predictions.jsonl')
)
assert len(action_baseline_predictions) == 1, action_baseline_predictions
perception_baseline_predictions = REPO_ROOT / 'experiments/exp002_qwen35_2b_molmoweb/artifacts/validation_probes/step_1875/' 'predictions.jsonl'
perception_latest_predictions = REPO_ROOT / 'outputs/experiment4/evaluation/' / 'perception-development-step-1875' / 'predictions.jsonl'
checkpoint_paths = {
    'baseline': perception_baseline_predictions,
    'latest': perception_latest_predictions,
}
action_paths = {
    'baseline': action_baseline_predictions[0],
    'latest': action_predictions_path,
}
payload = build_probe_dashboard(
    checkpoint_paths,
    checkpoint_labels={'baseline': 'EXP002 parent', 'latest': 'EXP004 · step 1875'},
    latest_step=1875, action_prediction_paths=action_paths,
)
dashboard_root = REPO_ROOT / 'outputs/experiment4/dashboard'
write_dashboard_json(payload, dashboard_root / 'qa-probe.json')
copied = copy_action_dashboard_images(
    payload['action'], prepared, dashboard_root / 'images/action'
)
print({'event': 'dashboard_export_complete', 'images_copied': copied}, flush=True)


In [ ]:
baseline_paths = list(Path('/kaggle/input').rglob('action-exp002/metrics.json'))
assert len(baseline_paths) == 1, baseline_paths
action_baseline = json.loads(baseline_paths[0].read_text())
perception_baseline_path = REPO_ROOT / 'experiments/exp002_qwen35_2b_molmoweb/artifacts/validation_probes/step_1875/summary.json'
perception_baseline = json.loads(perception_baseline_path.read_text())['primary_metrics']
gate = {
  'step': 1875,
  'action_baseline': action_baseline,
  'action_candidate': action_metrics,
  'perception_baseline': perception_baseline,
  'perception_candidate': perception['primary_metrics'],
}
gate['perception_regressions'] = {
  key: {'baseline': perception_baseline[key], 'candidate': perception['primary_metrics'][key]}
  for key in ('qa_answer_accuracy', 'grounding_click_accuracy')
  if perception['primary_metrics'][key] < perception_baseline[key] - 0.03
}
gate['action_regressions'] = {
  key: {'baseline': action_baseline[key], 'candidate': action_metrics[key]}
  for key in ('action_name_accuracy', 'click_inside_bbox_accuracy')
  if action_metrics[key] is not None and action_baseline[key] is not None
  and action_metrics[key] < action_baseline[key] - 0.02
}
gate['advance'] = not gate['perception_regressions'] and not gate['action_regressions']
gate_path = REPO_ROOT / 'outputs/experiment4/validation_gate.json'
gate_path.write_text(json.dumps(gate, indent=2) + '\n')
print({'event': 'validation_gate_complete', **gate}, flush=True)
